In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import json

plt.style.use("seaborn-v0_8")
sns.set_palette("Set2")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

ANIMAL_FILE = "data/animal_data_dirty1.csv"
MOVIES_FILE = "data/movies_metadata.csv"
CREDITS_FILE = "data/credits.csv"

# ============================================================
# ЗАДАНИЕ 1. Предварительный анализ данных. Обработка и принципы работы
# ============================================================

df = pd.read_csv(ANIMAL_FILE)

display(df.head())
print("Исходный размер датасета:", df.shape)

print("\nТипы данных и наличие пропусков:")
display(df.info())

print("\nОсновные числовые характеристики исходного датасета:")
display(df.describe(include="all"))

missing_before = df.isna().sum().sort_values(ascending=False)
print("\nКоличество пропусков по столбцам:")
display(missing_before)

missing_nonzero = missing_before[missing_before > 0]

plt.figure(figsize=(10, 5))
if not missing_nonzero.empty:
    missing_nonzero.plot(kind="bar", color="#4C72B0")
    plt.title("Количество пропусков по столбцам")
    plt.xlabel("Столбцы")
    plt.ylabel("Количество пропусков")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Пропуски в датасете не обнаружены.")

# try:
#     import missingno as msno
#     msno.matrix(df)
#     plt.title("Матрица пропусков")
#     plt.show()
# except Exception:
#     pass

print("Количество дубликатов:", df.duplicated().sum())

num_cols_initial = df.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols_initial) > 0:
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=df[num_cols_initial], orient="h")
    plt.title("Диаграммы размаха числовых признаков до очистки")
    plt.tight_layout()
    plt.show()

df_clean = df.copy()

for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        df_clean[col] = df_clean[col].astype(str).str.strip()
        df_clean[col] = df_clean[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        converted = pd.to_numeric(df_clean[col], errors="coerce")
        if converted.notna().sum() >= max(5, int(0.7 * len(df_clean))):
            df_clean[col] = converted

for col in df_clean.columns:
    if pd.api.types.is_numeric_dtype(df_clean[col]):
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    else:
        mode_value = df_clean[col].mode(dropna=True)
        if len(mode_value) > 0:
            df_clean[col] = df_clean[col].fillna(mode_value[0])

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
outlier_counts = {}

for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = df_clean[col].between(lower, upper)
    outlier_counts[col] = int((~mask).sum())
    df_clean.loc[~mask, col] = df_clean[col].median()

print("\nКоличество выбросов по числовым столбцам:")
display(pd.Series(outlier_counts).sort_values(ascending=False))

print("\nПроверка на противоречия:")

contradictions = []

for col in df_clean.columns:
    if pd.api.types.is_numeric_dtype(df_clean[col]):
        if (df_clean[col] < 0).any():
            contradictions.append(f"{col}: есть отрицательные значения")

for col in df_clean.select_dtypes(include="object").columns:
    if df_clean[col].nunique(dropna=True) <= 1:
        contradictions.append(f"{col}: слишком мало уникальных значений")

if len(contradictions) == 0:
    print("Явных противоречий по базовым правилам не найдено.")
else:
    for item in contradictions:
        print("-", item)

print("\nРазмер очищенного датасета:", df_clean.shape)
print("\nОсновные числовые характеристики очищенного датасета:")
display(df_clean.describe(include="all"))

if len(numeric_cols) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=df[numeric_cols], orient="h", ax=axes[0])
    axes[0].set_title("До очистки")
    sns.boxplot(data=df_clean[numeric_cols], orient="h", ax=axes[1])
    axes[1].set_title("После очистки")
    plt.tight_layout()
    plt.show()

if len(numeric_cols) > 0:
    plt.figure(figsize=(10, 5))
    corr = df_clean[numeric_cols].corr(numeric_only=True)
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Корреляция числовых признаков после очистки")
    plt.tight_layout()
    plt.show()

print("\nВыводы по заданию 1:")
print(f"1. Исходный датасет содержал пропуски, дубликаты и выбросы.")
print(f"2. После очистки размер датасета стал {df_clean.shape[0]} строк и {df_clean.shape[1]} столбцов.")
print("3. Пропуски были заполнены: числовые признаки медианой, категориальные — модой.")
print("4. Дубликаты удалены, выбросы по числовым признакам обработаны методом IQR.")
print("5. Датасет приведен к более удобному и чистому виду для дальнейшего анализа.")

# ============================================================
# ЗАДАНИЕ 2. The Movies Dataset: наименование фильма, режиссер, жанр
# ============================================================

movies = pd.read_csv(MOVIES_FILE)
credits = pd.read_csv(CREDITS_FILE)

print("Размер movies_metadata:", movies.shape)
print("Размер credits:", credits.shape)
display(movies.head(2))
display(credits.head(2))

# Берем только нужные столбцы из movies_metadata
movies_simple = movies[['id', 'title', 'vote_average', 'genres']].copy()
movies_simple['id'] = pd.to_numeric(movies_simple['id'], errors='coerce')
movies_simple = movies_simple.dropna(subset=['id'])

credits['id'] = pd.to_numeric(credits['id'], errors='coerce')
credits = credits.dropna(subset=['id'])

print("После очистки ID:")
print("movies:", movies_simple.shape)
print("credits:", credits.shape)

df_merged = movies_simple.merge(credits[['id', 'crew']], on='id', how='inner')
print("После слияния:", df_merged.shape)

def get_director(crew_str):
    if pd.isna(crew_str):
        return np.nan
    try:
        crew = ast.literal_eval(crew_str)
        for person in crew[:5]:  # первые 5 человек
            if person.get('job') == 'Director':
                return person.get('name')
    except:
        pass
    return np.nan

def get_genres(genres_str):
    if pd.isna(genres_str):
        return np.nan
    try:
        genres = ast.literal_eval(genres_str)
        return ', '.join([g['name'] for g in genres[:3] if 'name' in g])
    except:
        return np.nan

df_merged['director'] = df_merged['crew'].apply(get_director)
df_merged['genre'] = df_merged['genres'].apply(get_genres)

result_df = df_merged[['title', 'director', 'genre', 'vote_average']].dropna(subset=['vote_average'])
result_df.columns = ['movie_title', 'director', 'genre', 'vote_average']

print("Итоговый датасет:", result_df.shape)
display(result_df.head())

# Статистика по режиссерам (ТОЛЬКО с >=3 фильмами)
director_stats = result_df.dropna(subset=['director']).groupby('director')['vote_average'].agg(['mean', 'median', 'count']).reset_index()
director_stats = director_stats[director_stats['count'] >= 3].sort_values('mean', ascending=False)

# Статистика по жанрам (ТОЛЬКО с >=10 фильмами)
genre_stats = (
    result_df.dropna(subset=['genre'])
    .assign(genre=lambda x: x['genre'].str.split(', '))
    .explode('genre')
    .groupby('genre')['vote_average']
    .agg(['mean', 'median', 'count'])
    .reset_index()
)
genre_stats = genre_stats[genre_stats['count'] >= 10].sort_values('mean', ascending=False)

print("\nТОП-10 РЕЖИССЕРОВ (минимум 3 фильма):")
display(director_stats.head(10))

print("\nТОП-10 ЖАНРОВ (минимум 10 фильмов):")
display(genre_stats.head(10))

print("\nХУДШИЕ РЕЖИССЕРЫ (минимум 3 фильма):")
display(director_stats.tail(5))

print("\nХУДШИЕ ЖАНРЫ (минимум 10 фильмов):")
display(genre_stats.tail(5))

# Визуализация
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.barplot(data=director_stats.head(8), x='mean', y='director', ax=axes[0,0], palette='viridis')
axes[0,0].set_title('Лучшие режиссеры (≥3 фильма)')
axes[0,0].set_xlabel('Средний рейтинг')

sns.barplot(data=genre_stats.head(8), x='mean', y='genre', ax=axes[0,1], palette='plasma')
axes[0,1].set_title('Лучшие жанры (≥10 фильмов)')
axes[0,1].set_xlabel('Средний рейтинг')

sns.histplot(result_df['vote_average'], bins=20, kde=True, ax=axes[1,0], color='coral')
axes[1,0].set_title('Распределение всех рейтингов')
axes[1,0].set_xlabel('Рейтинг')

director_count = director_stats['count'].value_counts().sort_index()
if len(director_count) > 0:
    director_count.plot(kind='bar', ax=axes[1,1], color='lightblue')
    axes[1,1].set_title('Количество режиссеров по числу фильмов')
    axes[1,1].set_xlabel('Количество фильмов')

plt.tight_layout()
plt.show()

print("\nВыводы по заданию 2:")
print(f"1. Создан датасет из {len(result_df)} фильмов с названием, режиссером, жанром, рейтингом.")
print(f"2. Найдено {len(director_stats)} режиссеров с ≥3 фильмами.")
print(f"3. Найдено {len(genre_stats)} жанров с ≥10 фильмами.")
print("4. Построены рейтинги и визуализации.")

EmptyDataError: No columns to parse from file